In [1]:
import os
import sys
# Agregar el directorio raíz del proyecto al path
root_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
    
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *   
from services.utilidades import *
from services.comprobar_netcdf import *
from services.carga_meses_de_telemetria import cargar_telemetria_oleaje
from services.listado_de_boyas import obtener_boyas

# Recargar módulos locales (ejecutar esta celda cuando cambies código en services/)
import importlib
import sys

# Lista de módulos locales a recargar
modulos_locales = [
    'services.carga_de_datos',
    'services.DataFrame_maker',
    'services.prueba',
    'services.utilidades',
    'services.get_tspan',
    'services.polar2uv',
    'services.uv2polar'
]

for modulo in modulos_locales:
    if modulo in sys.modules:
        importlib.reload(sys.modules[modulo])
        print(f"✓ Recargado: {modulo}")
    else:
        print(f"- No cargado: {modulo}")

# Re-importar después de recargar
from services.DataFrame_maker import *
from services.InteractivePlotter import InteractivePlotterTk
from services.carga_de_datos import *
from services.utilidades import *
from services.comprobar_netcdf import *
from services.carga_meses_de_telemetria import cargar_telemetria_oleaje
from services.listado_de_boyas import obtener_boyas

print("\n¡Módulos locales recargados!")

✓ Recargado: services.carga_de_datos
✓ Recargado: services.DataFrame_maker
- No cargado: services.prueba
✓ Recargado: services.utilidades
✓ Recargado: services.get_tspan
✓ Recargado: services.polar2uv
✓ Recargado: services.uv2polar

¡Módulos locales recargados!


In [2]:
# Configuraciones
mes = "9"
anio = "2026"

In [3]:
dict_de_boyas = {}

# listar_funciones_de_carga()
# Para ver la numercación de las boyas ve al archivo listado_de_boyas.py
# listado = obtener_boyas()

### GRUPOS DE BOYAS VALIDOS ###

In [ ]:
# # # "BOT1-01-T80", "BOT1-10-T40", "BMT3-03-T80", , "BMT3-10-T45",
# boyas = obtener_boyas(eleccion = [1,8,13,16], estado="todos")

# # BMT2-01-T20","BMT3-03-T80","BMT3-10-T45"
# boyas = obtener_boyas(eleccion = [12,13,16], estado="todos")

# # BOT1-07-T80","BOT1-10-T40"
# boyas = obtener_boyas(eleccion = [6, 8], estado="todos")

# #"BOT1-03-T50","BOT1-04-T80","BMT3-04-T45","BOT2-01-T20","BOT2-02-T20",
# boyas = obtener_boyas(eleccion = [2,3,10,11,14], estado="todos")

# #"BOT1-04-T80","BOT1-05-T50","BOT1-06-T50","BMT3-04-T45",
# boyas = obtener_boyas(eleccion = [3,4,5,14], estado="todos")

# # BOT1-06-T50",BMT3-11-T1000",
# boyas = obtener_boyas(eleccion = [5,17], estado="todos")

# #"BOT1-09-T50",
# boyas = obtener_boyas(eleccion = [7], estado="todos")

# #"BOT1-11-T100",
# boyas = obtener_boyas(eleccion = [9], estado="todos")

### ELECCION DE UNA BOYA PARA REVISIÓN CRUDOS VS VALIDADOS

In [5]:
# # Elija una boya
# eleccion = [1]
# boyas = obtener_boyas(eleccion = eleccion, estado="todos")

### NO TOCAR

In [6]:
print("HAS ELEGIDO LA(S) BOYA(S)")
for boya, _ in boyas.items():
    print(f"- {boya}")

HAS ELEGIDO LA(S) BOYA(S)
- BOT1-11-T100


In [7]:
boyas

{'BOT1-11-T100': ['crudo', 'validado']}

In [8]:
for boya, estados in boyas.items():
    for estado in estados:
        key = f"{boya}_{estado}"
        tmp = cargar_telemetria_oleaje(boya, [int(anio)], [int(mes)], estado)
        if tmp.empty:
            print(f"**** LA BOYA {boya} NO TIENE DATOS {estado} ****")
            continue
        dict_de_boyas[key] = tmp[["tspan","Hs","Hmax","Tp","dir","vardir"]]
        

In [9]:
# Crear el diccionario de boyas con los dataframes cargados
df = crear_dataframe_de_boyas(dict_de_boyas)

Index(['tspan', 'Hs_BOT1-11-T100_crudo', 'Hmax_BOT1-11-T100_crudo',
       'Tp_BOT1-11-T100_crudo', 'dir_BOT1-11-T100_crudo',
       'vardir_BOT1-11-T100_crudo', 'Hs_BOT1-11-T100_validado',
       'Hmax_BOT1-11-T100_validado', 'Tp_BOT1-11-T100_validado',
       'dir_BOT1-11-T100_validado', 'vardir_BOT1-11-T100_validado'],
      dtype='str')


In [10]:
app = InteractivePlotterTk(df)
# Iniciar el bucle de la aplicación
app.root.mainloop()

# NOTAS PENDIENTES:
# - Arreglar que al eliminar puntos para que no tenga límite el historial de deshacer
# - Arreglar para que solo se grafiquen las columnas elegidas por el usuario
# - Agregar una función que grafique las variables de Rap y Dir --> sería algo como que df.iloc[idx,col][prof]
# - Arreglar el caso de cuando el usuario elimina puntos de Rap o Dir colocar U y V como nans --> df.iloc[idx,col][prof] = np.nan
# - Agregar la funcionalidad de guardar el DataFrame corregido en un archivo .csv o .xlsx, pkl
# - Agregar la funcionalidad de devolver puntos eliminados (no deshacer la última acción, sino devolver puntos específicos)

In [11]:
# from services.guardar_puntos_eliminados import guardar_puntos_eliminados
# nombre_de_archivo = "borrar_puntos"
# guardar_puntos_eliminados(app, ruta_a_carpeta, nombre_de_archivo)

In [12]:
# from pickle import load
# ruta = os.path.join(ruta_a_carpeta, "borrar_puntos.pkl")
# with open(ruta, 'rb') as f:
#     puntos_eliminados = load(f)
    
# puntos_eliminados